# Cross-renderer diagnostic: frozen 60k CRNN

200 word forms × 3 fonts = 600 images. All forms are excluded from every original train/development/test manifest. This is a development diagnostic, not a new final test. The checkpoint is unchanged epoch 13. Font overlap with tyotakuki is unknown.

The local word list is attributed to Wiktionary in the original repository README. Font identities and hashes, exclusions, labels and rendering settings are in `data/cross_renderer_v1/SOURCE.json`. Native Pillow/RAQM shaping replaces the old generator's Arabic reshaper; the old generator and its existing modifications were preserved.

Images use left-to-right horizontal Manchu shaping, fit into a white 480×64 canvas without distortion. The unchanged model performs its own crop and 32×320 resampling. Calibration examples were visually inspected before scoring. Labels have not been independently verified by a Manchu reader.

The eligible local vocabulary did not provide 13+ character words after exclusions; actual length bins are 1–4, 5–8 and 9–12. Results cannot establish long-word generalization.


In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
if not (ROOT / "research_runs").exists():
    ROOT = ROOT / "research"
report = json.loads((ROOT / "research_runs/cross_renderer_v1/results.json").read_text())
print("Frozen checkpoint epoch:", report["checkpoint_epoch"])
for font, metrics in report["by_font"].items():
    print(f"{font}: {metrics['exact_matches']}/{metrics['n']} words; accuracy={metrics['word_accuracy']:.1%}; CER={metrics['cer']:.2%}")
print("Overall:", report["overall"])
print("By length:", json.dumps(report["by_length"], indent=2))


## Calibration (separate words, inspected before scoring)
![Calibration (separate words, inspected before scoring)](data/cross_renderer_v1/calibration.png)


## Correct and incorrect predictions
![Correct and incorrect predictions](research_runs/cross_renderer_v1/examples.png)


## Interpretation and next step
Accuracy varies sharply by rendering style. This supports testing mixed-font adaptation before changing the recognizer architecture. Source vocabulary, labels, font and rasterization are confounded; these results do not isolate a single causal factor.

Audit a few labels and contextual glyph forms with a competent reader, then prepare separate mixed-font training and word-disjoint development data, excluding these diagnostic words and all original evaluation forms. Fine-tune a copy of epoch 13; compare against the frozen baseline on development and original unseen-word development to detect forgetting. Reserve a new independent final evaluation before model selection. No adaptation has been run.

## Reproduction
The experiment was run with `manchu/bin/python experiments/cross_renderer.py prepare` followed by `manchu/bin/python experiments/cross_renderer.py evaluate` from `research/`. Each command refuses to overwrite its existing output. Preserve v1; use new versioned paths for further experiments. Dependencies: torch, Pillow with RAQM, numpy and fonttools. Existing source archives and checkpoints were read only.
